# Chapter 15 &mdash; PCP as a Stepping Stone: Grammar Ambiguity and Beyond

**Concept 4 of the Chapter 15 decomposition:** *PCP as a Stepping Stone: Grammar Ambiguity and Other Undecidable Problems*

Reduce PCP to CFG ambiguity via a gadget; likewise predicate-logic validity and pointer analysis.

---

*Run on Colab:* [![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter15/Concept-PCP-As-Stepping-Stone/Concept-PCP-As-Stepping-Stone.ipynb)
*Or run locally from inside a Jove checkout.*

## 0. Setup

In [ ]:
# Run me first.  Works on Colab and on a local Jove checkout.
import os, subprocess, sys

def _git(*a):
    r = subprocess.run(('git',) + a, capture_output=True, text=True)
    return r.stdout.strip() if r.returncode == 0 else ''

REPO = 'https://github.com/ganeshutah/Jove'
try:                       # ---- Colab: clone once, pull thereafter ----
    import google.colab
    was = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD') if os.path.isdir('Jove') else ''
    if os.path.isdir('Jove') and not was:
        print('Jove: WARNING ./Jove exists but is not a git checkout -- left as is')
    elif was:
        _git('-C', 'Jove', 'pull', '-q', '--ff-only')
        now = _git('-C', 'Jove', 'rev-parse', '--short', 'HEAD')
        if now and now != was:
            print('Jove: PULLED  %s -> %s' % (was, now))
            print(_git('-C', 'Jove', 'log', '--oneline', was + '..' + now))
        else:
            print('Jove: PULLED  already current at %s' % (now or was))
    else:
        _git('clone', '-q', REPO, 'Jove')
        print('Jove: CLONED  at %s' % (_git('-C', 'Jove', 'rev-parse',
                                             '--short', 'HEAD') or '?'))
    JOVE = 'Jove'
except ImportError:        # ---- local: the checkout above Chapter<N>/ ----
    JOVE = next((p for p in ('../..', '../../..', '..', '.')
                 if os.path.isdir(os.path.join(p, 'jove'))), '../..')
    print('Jove: LOCAL   checkout at %s'
          % (_git('-C', JOVE, 'rev-parse', '--short', 'HEAD') or '?'))
sys.path.insert(0, JOVE)

# A session can already hold an OLDER jove in sys.modules.  The pull above
# updates the files on disk, but `import` would hand back the cached module --
# so a fixed library still behaves like the broken one.  Drop them first.
for _m in [k for k in list(sys.modules) if k == 'jove' or k.startswith('jove.')]:
    del sys.modules[_m]

from jove.Def_md2mc      import *
from jove.DotBashers     import *
from jove.Def_DFA        import *
from jove.Def_NFA        import *
from jove.LangDef        import *

import jove; print('Jove loaded from', list(jove.__path__)[0])

## 1. The idea


Once PCP is undecidable, it becomes a **source** for other undecidability proofs &mdash;
often more convenient than $A_{TM}$, because its structure is combinatorial rather than
computational.

**CFG ambiguity.** Given dominoes $(t_i,b_i)$, build the grammar
$$S \to A \mid B,\qquad A \to t_i A c_i \mid t_i c_i,\qquad B \to b_i B c_i \mid b_i c_i$$
with fresh index symbols $c_i$. A string derivable **both** ways spells out a top
sequence and a bottom sequence with the *same* index list &mdash; a PCP solution. So the
grammar is ambiguous iff the instance is solvable.

Others in the same family: **predicate-logic validity**, **pointer analysis**, the
**domino tiling** of the plane, and the **intersection-emptiness** of two CFGs.

## 2. Definitions

### PCP, and the ambiguity gadget

In [ ]:
# --- a Post Correspondence solver, bounded by tile count ----------------
# An instance is a list of (top, bottom) dominoes.  A solution is a
# non-empty sequence of indices whose concatenated tops equal its bottoms.
def pcp_search(tiles, maxlen=8):
    from collections import deque
    # a partial solution is (indices, top, bottom); one side is a prefix
    # of the other, or the partial is dead
    dq = deque([((i,), t, b) for i, (t, b) in enumerate(tiles)])
    while dq:
        idx, top, bot = dq.popleft()
        if top == bot:
            return list(idx)
        if len(idx) >= maxlen:
            continue
        if not (top.startswith(bot) or bot.startswith(top)):
            continue                                  # dead: they diverge
        for j, (t, b) in enumerate(tiles):
            dq.append((idx + (j,), top + t, bot + b))
    return None

def pcp_check(tiles, sol):
    top = ''.join(tiles[i][0] for i in sol)
    bot = ''.join(tiles[i][1] for i in sol)
    return top == bot, top, bot

def show_tiles(tiles):
    print("   " + "  ".join("[%s/%s]" % t for t in tiles))
# --- a tiny CFG toolkit -------------------------------------------------
# A grammar is a dict with keys N (nonterminals), Sigma (terminals),
# S (start symbol) and P (productions: nonterminal -> list of RHS tuples).
# A right-hand side is a tuple of one-character symbols; () is epsilon.
# By convention UPPERCASE single letters are nonterminals.

def mkg(rules, start='S'):
    N = set(rules)
    P = {A: [tuple(r) for r in rhs] for A, rhs in rules.items()}
    Sigma = {c for rhs in P.values() for r in rhs for c in r if c not in N}
    return dict(N=N, Sigma=Sigma, S=start, P=P)

def show(G):
    print("N     =", sorted(G['N']))
    print("Sigma =", sorted(G['Sigma']))
    print("S     =", G['S'])
    for A in sorted(G['P']):
        alts = ' | '.join((''.join(r) if r else "''") for r in G['P'][A])
        print("   %s -> %s" % (A, alts))

def derivable(G, maxlen):
    # least fixed point: for each nonterminal, every terminal string of
    # length <= maxlen it derives.  Far cheaper than searching sentential
    # forms, and it terminates because the sets only grow and are bounded.
    T = {A: set() for A in G['N']}
    def spans(r):
        acc = {''}
        for x in r:
            src = T[x] if x in T else {x}
            acc = {a + b for a in acc for b in src if len(a) + len(b) <= maxlen}
            if not acc: break
        return acc
    changed = True
    while changed:
        changed = False
        for A in G['P']:
            for r in G['P'][A]:
                for w in spans(r):
                    if w not in T[A]:
                        T[A].add(w); changed = True
    return T

def language(G, maxlen):
    return sorted(derivable(G, maxlen)[G['S']], key=lambda s: (len(s), s))

def _spans(G, w, cap=None):
    # Bottom-up, shortest span first, so a span never depends on a LONGER
    # one.  Within a span we iterate |N|+1 times, which is enough to close
    # unit rules (A -> B) and epsilon rules.  Doing it top-down with a
    # "cycle guard" silently poisons the memo table, so we do not.
    n, N, P = len(w), G['N'], G['P']
    tab = {}                       # (A, i, j) -> count, or list of trees
    def get(sym, i, j):
        if sym not in N:
            if j == i + 1 and w[i] == sym:
                return 1 if cap is None else [sym]
            return 0 if cap is None else []
        return tab.get((sym, i, j), 0 if cap is None else [])
    def seqv(r, i, j):
        if not r:
            if i != j: return 0 if cap is None else []
            return 1 if cap is None else [()]
        acc = 0 if cap is None else []
        for k in range(i, j + 1):
            a = get(r[0], i, k)
            if not a: continue
            b = seqv(r[1:], k, j)
            if not b: continue
            if cap is None:
                acc += a * b
            else:
                for h in a:
                    for t in b:
                        acc.append((h,) + tuple(t))
                        if len(acc) >= cap: return acc
        return acc
    for length in range(0, n + 1):
        for i in range(0, n - length + 1):
            j = i + length
            for _ in range(len(N) + 1):
                grew = False
                for A in P:
                    v = []
                    for r in P[A]:
                        x = seqv(r, i, j)
                        if cap is None:
                            v.append(x)
                        else:
                            v += [(A,) + tuple(t) for t in x]
                            if len(v) >= cap: v = v[:cap]; break
                    v = sum(v) if cap is None else v
                    old = tab.get((A, i, j), 0 if cap is None else [])
                    if (v != old) if cap is None else (len(v) != len(old)):
                        tab[(A, i, j)] = v; grew = True
                if not grew: break
    return get(G['S'], 0, n)

def nparses(G, w):
    return _spans(G, w, cap=None)

def parse_trees(G, w, cap=8):
    return _spans(G, w, cap=cap)

def yield_of(t):
    return t if isinstance(t, str) else ''.join(yield_of(c) for c in t[1:])

def show_tree(t, ind=0):
    if isinstance(t, str):
        print("%s'%s'" % ('  ' * ind, t)); return
    print("%s%s" % ('  ' * ind, t[0]))
    for c in t[1:]: show_tree(c, ind + 1)

def leftmost(G, w):
    # the leftmost derivation read off one parse tree
    ts = parse_trees(G, w, cap=1)
    if not ts: return None
    steps, form = [], [G['S']]
    def expand(t, pos):
        # t is the subtree rooted at the nonterminal currently at `pos`
        if isinstance(t, str): return pos + 1
        kids = [c if isinstance(c, str) else c[0] for c in t[1:]]
        form[pos:pos+1] = kids
        steps.append(''.join(form) or "''")
        p = pos
        for c in t[1:]:
            p = expand(c, p)
        return p
    steps.append(G['S'])
    expand(ts[0], 0)
    return steps


def pcp_to_grammar(tiles, idx='xyzw'):
    # S -> A | B ;  A -> t_i A c_i | t_i c_i ;  B -> b_i B c_i | b_i c_i
    A = [tiles[i][0] + 'A' + idx[i] for i in range(len(tiles))] + \
        [tiles[i][0] + idx[i] for i in range(len(tiles))]
    B = [tiles[i][1] + 'B' + idx[i] for i in range(len(tiles))] + \
        [tiles[i][1] + idx[i] for i in range(len(tiles))]
    return mkg({'S': ["A", "B"], 'A': A, 'B': B})

### Two instances: one solvable, one not (as far as we can search)

In [ ]:
SOLVABLE = [('a', 'ab'), ('ba', 'a')]     # [0, 1] gives 'aba' on both rows
UNSOLVED = [('ab', 'aba'), ('bb', 'aa')]  # every tile's bottom is >= its top

<!-- nav-strip -->

---

&larr;&nbsp;[Ch15&nbsp;3.&nbsp;Tile Construction: Simulating a TM Through a Peephole](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter15/Concept-Tile-Construction/Concept-Tile-Construction.ipynb) &nbsp;&middot;&nbsp; [**Chapter 15** index](https://github.com/ganeshutah/Jove/blob/master/Chapter15/README.md) &nbsp;&middot;&nbsp; [Ch15&nbsp;5.&nbsp;$A_{TM}$ is Undecidable: the Diagonalization Proof](https://colab.research.google.com/github/ganeshutah/Jove/blob/master/Chapter15/Concept-A-TM-Diagonalization/Concept-A-TM-Diagonalization.ipynb)&nbsp;&rarr;

---

## 3. Tests

The solvable instance, and its solution.

In [ ]:
show_tiles(SOLVABLE)
sol = pcp_search(SOLVABLE, maxlen=6)
ok, top, bot = pcp_check(SOLVABLE, sol)
print("solution", sol, "->", top, "==", bot, ":", ok)
assert ok

Its grammar is **ambiguous** &mdash; the same string has an $A$-tree and a $B$-tree.

In [ ]:
G = pcp_to_grammar(SOLVABLE)
show(G)
L = language(G, 8)
amb = [w for w in L if nparses(G, w) > 1]
print("\nstrings with more than one parse :", amb[:5])
assert amb
w = amb[0]
print("  %r has %d parse trees" % (w, nparses(G, w)))

And the ambiguous string **encodes the solution**.

In [ ]:
w = amb[0]
print("ambiguous string :", w)
tops = ''.join(SOLVABLE[i][0] for i in sol)
bots = ''.join(SOLVABLE[i][1] for i in sol)
print("solution tops    :", tops)
print("solution bottoms :", bots)
print("\nThe index suffix is the SAME on both derivations, which is exactly")
print("the requirement that the two rows use the same domino sequence.")

A grammar with **no** ambiguity, up to the search bound.

In [ ]:
G2 = pcp_to_grammar(UNSOLVED)
L2 = language(G2, 8)
amb2 = [w for w in L2 if nparses(G2, w) > 1]
print("strings generated :", len(L2))
print("ambiguous ones    :", amb2)
assert not amb2
print("\nNo ambiguity found -- consistent with finding no PCP solution.")
print("And just as inconclusive: we searched a bound, not the whole language.")

The family of problems PCP settles.

In [ ]:
VIA_PCP = [("CFG ambiguity",              "the gadget above"),
           ("CFG intersection emptiness", "two grammars, one for each row"),
           ("predicate-logic validity",   "encode dominoes as axioms"),
           ("tiling of the plane",        "Wang tiles, directly"),
           ("some pointer analyses",      "alias questions encode matching")]
for a, b in VIA_PCP: print("  %-28s %s" % (a, b))
print("\nEach is easier FROM PCP than from A_TM, because PCP is already")
print("combinatorial -- there is no computation left to encode.")

## 4. Exercises


1. Check the gadget: why do the index symbols have to be **fresh**?
2. Reduce PCP to "do these two CFGs generate a common string?"
3. Which is the easier source for a new undecidability proof, $A_{TM}$ or PCP? Why?

In [ ]:
# Your work for the exercises above.

## 5. Where next

In [ ]:
# Previous / next, and a search box for all 252 concepts.
# Type a chapter (Chapter7, ch7) or words from a title (pumping, subset).
#
# Following a link opens a NEW Colab runtime. To pull another concept's
# definitions into THIS session instead:  load_here('Chapter7/Concept-...')
import os, sys
try:                       # usually already done by the Setup cell
    import jove
except ModuleNotFoundError:
    _p = next((p for p in ('Jove', '../..', '../../..', '..', '.')
               if os.path.isdir(os.path.join(p, 'jove'))), None)
    if _p:
        sys.path.insert(0, _p)
try:
    from jove.Nav import nav, load_here
    nav(here='Chapter15/Concept-PCP-As-Stepping-Stone')
except ModuleNotFoundError:
    print('Jove is not on the path yet.')
    print('Run the Setup cell at the top of this notebook, then re-run this one.')